# 多Agent辩论和协作

跑N个模型实例，独立地生成答案。然后迭代的批判其他答案N轮直到手链。能够改善事实性、规则遵循、推理。稀疏拓扑在成本上胜过全连接网络。

## 问题描述

Self-Refine 是一个模型批判自身，有群体思维的风险。CRITIC 将批判锚定给外部工具，不总是可用。Debate 引入了第三种模式：多个实例，交叉批判，靠分歧来收敛。

## 基本概念

Society of Minds：
- N个模型实例针对同一个问题独立提出答案。
- 在R轮循环中，每个模型读其他人的答案，然后做出批判
- 模型基于批判内容更新答案
- R轮循环后，返回收敛的答案

随着agent和轮次上升，难问题上的准确率上升。
另外交叉使用模型比只用单个模型辩论效果好。

### 稀疏拓扑

全连接 N * R 次 LLM 调用来批判，开销比较大。事实表明稀疏拓扑（不对所有人批判，挑一部分）能够在降低token花费的前提下保住准确性。

常见的稀疏形状
|拓扑|辩论面|
|---|---|
|环|只看左右邻居|
|星形|边缘->中心->广播|
|随机|每轮随机抽k个人|

### 优缺点
|优点|缺点|
|---|---|
|事实性，交叉辩论降低幻觉|延迟上升|
|规则遵循，交叉辩论发现违反|花费上升|
|开放推理，多种框定收窄到正确答案|不要用于简单事实查询|

### 2026 年的实践实例

- Anthropic. 编排-工作者模式。 辩论的变体，加上一次合成步骤。
- LangGraph。 主管模式。 中心路由+专家agents 可以实现辩论。
- OpenAI Agent SDK。  agents 来回交接实现迭代批判
- Multi-agent evals。 把辩论+evaluator+optimizer组合起来获取评估信号。

### 什么时候这种模式出错
- 收敛坍塌。  所有的agents收敛到第一个错误答案，用强制分歧轮来缓解。
- Hub 失败。 星型拓扑里，一个坏掉的hub会污染其他所有人。旋转或者使用多个hub。
- 提示词同质。 所有的agents使用相同的提示词，产出的答案也相同。使用不同的提示词或者不同的模型。

# 开始编码

对应本章核心：**Society of Minds（提案→交叉批判→更新×R）**、**稀疏拓扑（环/星/随机 k）**、**强制分歧轮防收敛坍塌**、**异构 persona 防提示词同质**。  
先用脚本化玩具跑通拓扑与强制分歧；再用 **LangGraph + DeepSeek** 做生产辩论（不硬凑 PyTorch）。无 `DEEPSEEK_API_KEY` 则生产示例 SKIP。


## 1. 教学玩具：Debate 运行时

- **拓扑**：`full` / `ring` / `star` / `random_k` → 每轮每人只读邻居答案。
- **轮次**：独立提案 →（可选强制分歧）→ 普通批判更新 → 多数票汇总。
- **强制分歧**：本轮禁止「同意」，必须给反方案。
- **异构**：每个 agent 不同 persona / 初始偏置。


In [7]:
from __future__ import annotations

import random
from collections import Counter
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

Topology = Literal["full", "ring", "star", "random_k"]


@dataclass
class AgentState:
    """单个辩论 agent 的状态。"""

    agent_id: int
    persona: str
    answer: str = ""
    critiques: list[str] = field(default_factory=list)


@dataclass
class RoundLog:
    """一轮辩论日志。"""

    round_idx: int
    mode: Literal["propose", "forced_dissent", "critique"]
    reads: int
    answers: dict[int, str]


@dataclass
class DebateResult:
    """整场辩论结果。"""

    final_answer: str
    agents: list[AgentState]
    logs: list[RoundLog]
    total_reads: int


def neighbors(
    n: int,
    i: int,
    topology: Topology,
    *,
    k: int = 2,
    rng: random.Random | None = None,
) -> list[int]:
    """
    Args:
        n: agent 数。
        i: 当前 agent 下标。
        topology: 通信拓扑。
        k: random_k 时的邻居数。
        rng: 随机源（random_k 可复现）。

    Returns:
        ids: 本轮可读的其他 agent 下标。
    """
    if n <= 1:
        return []
    others = [j for j in range(n) if j != i]
    if topology == "full":
        return others
    if topology == "ring":
        return [(i - 1) % n, (i + 1) % n]
    if topology == "star":
        # hub=0：边缘只读 hub；hub 读所有边缘
        if i == 0:
            return others
        return [0]
    if topology == "random_k":
        r = rng or random.Random(0)
        kk = min(k, len(others))
        return r.sample(others, kk)
    raise ValueError(f"unknown topology: {topology}")


PolicyFn = Callable[[AgentState, list[tuple[int, str]], str, str], str]


@dataclass
class DebateRuntime:
    """Society of Minds 玩具运行时。"""

    n: int
    personas: list[str]
    topology: Topology = "ring"
    k: int = 2
    seed: int = 0
    policy: PolicyFn | None = None

    def __post_init__(self) -> None:
        if len(self.personas) != self.n:
            raise ValueError("personas length must equal n")
        self.rng = random.Random(self.seed)
        self.policy = self.policy or default_scripted_policy

    def run(
        self,
        question: str,
        *,
        rounds: int = 2,
        forced_dissent_rounds: int = 1,
        initial_answers: list[str] | None = None,
    ) -> DebateResult:
        """
        Args:
            question: 辩论问题。
            rounds: 普通批判轮数（强制分歧轮另计）。
            forced_dissent_rounds: 强制分歧轮数（插在提案后）。
            initial_answers: 可选预置初始答案（测坍塌用）。

        Returns:
            result: 最终答案 + 日志。
        """
        agents = [
            AgentState(agent_id=i, persona=self.personas[i]) for i in range(self.n)
        ]
        logs: list[RoundLog] = []
        total_reads = 0

        # 1) 独立提案
        for a in agents:
            if initial_answers is not None:
                a.answer = initial_answers[a.agent_id]
            else:
                a.answer = self.policy(a, [], question, "propose")
        logs.append(
            RoundLog(
                round_idx=0,
                mode="propose",
                reads=0,
                answers={a.agent_id: a.answer for a in agents},
            )
        )

        round_idx = 1
        # 2) 强制分歧
        for _ in range(forced_dissent_rounds):
            reads, snapshot = self._critique_round(agents, question, mode="forced_dissent")
            total_reads += reads
            logs.append(
                RoundLog(
                    round_idx=round_idx,
                    mode="forced_dissent",
                    reads=reads,
                    answers=snapshot,
                )
            )
            round_idx += 1

        # 3) 普通批判
        for _ in range(rounds):
            reads, snapshot = self._critique_round(agents, question, mode="critique")
            total_reads += reads
            logs.append(
                RoundLog(
                    round_idx=round_idx,
                    mode="critique",
                    reads=reads,
                    answers=snapshot,
                )
            )
            round_idx += 1

        final = majority_vote([a.answer for a in agents])
        return DebateResult(final_answer=final, agents=agents, logs=logs, total_reads=total_reads)

    def _critique_round(
        self,
        agents: list[AgentState],
        question: str,
        *,
        mode: Literal["forced_dissent", "critique"],
    ) -> tuple[int, dict[int, str]]:
        """
        一轮：每人读邻居 → 更新答案（同步：基于本轮开始前快照）。

        Returns:
            reads: 本轮读边数。
            snapshot: 更新后答案。
        """
        before = {a.agent_id: a.answer for a in agents}
        reads = 0
        updates: dict[int, str] = {}
        for a in agents:
            nbrs = neighbors(self.n, a.agent_id, self.topology, k=self.k, rng=self.rng)
            peer_answers = [(j, before[j]) for j in nbrs]
            reads += len(peer_answers)
            new_ans = self.policy(a, peer_answers, question, mode)
            a.critiques.append(f"{mode}: saw {nbrs}")
            updates[a.agent_id] = new_ans
        for a in agents:
            a.answer = updates[a.agent_id]
        return reads, {a.agent_id: a.answer for a in agents}


def majority_vote(answers: list[str]) -> str:
    """
    Args:
        answers: 各 agent 最终答案。

    Returns:
        winner: 多数票；平票取字典序最小。
    """
    counts = Counter(answers)
    best = max(counts.values())
    cands = sorted([a for a, c in counts.items() if c == best])
    return cands[0]


def default_scripted_policy(
    agent: AgentState,
    peers: list[tuple[int, str]],
    question: str,
    mode: str,
) -> str:
    """
    脚本化策略：用 persona 偏置 + 邻居投票，演示拓扑与强制分歧。

    Returns:
        answer: 更新后的答案字符串。
    """
    # 题型：首都 / 算术 用固定候选；否则用 persona 标签
    q = question.lower()
    if "france" in q or "法国" in question:
        correct, wrong = "Paris", "Lyon"
    elif "2+2" in q or "二加二" in question:
        correct, wrong = "4", "5"
    else:
        correct, wrong = f"ok:{agent.persona}", f"bad:{agent.persona}"

    if mode == "propose":
        # 异构：偶数 agent 偏正确，奇数偏错误（可被辩论纠正）
        return correct if agent.agent_id % 2 == 0 else wrong

    peer_vals = [ans for _, ans in peers]
    if mode == "forced_dissent":
        # 禁止同意多数：必须给与邻居众数不同的答案
        if not peer_vals:
            return wrong if agent.answer == correct else correct
        maj = majority_vote(peer_vals)
        return wrong if maj == correct else correct

    # 普通批判：跟邻居多数；无邻居则保持
    if not peer_vals:
        return agent.answer
    return majority_vote(peer_vals + [agent.answer])


def count_edges(n: int, topology: Topology, *, k: int = 2, seed: int = 0) -> int:
    """
    统计一轮总读边数（衡量稀疏相对全连接的省钱程度）。

    Returns:
        edges: sum_i |neighbors(i)|。
    """
    rng = random.Random(seed)
    return sum(len(neighbors(n, i, topology, k=k, rng=rng)) for i in range(n))


print("DebateRuntime ready | topology + forced dissent")


DebateRuntime ready | topology + forced dissent


## 2. 玩具示例：边数、强制分歧、星形 hub


In [8]:
def demo_debate_toy() -> None:
    """断言稀疏边数、强制分歧打破假共识、星形 hub 影响面。"""
    n = 5
    personas = [f"p{i}" for i in range(n)]

    full_e = count_edges(n, "full")
    ring_e = count_edges(n, "ring")
    star_e = count_edges(n, "star")
    rand_e = count_edges(n, "random_k", k=2, seed=1)
    assert full_e == n * (n - 1)
    assert ring_e == 2 * n
    assert star_e == (n - 1) + (n - 1)  # hub 读 n-1，边缘各读 1
    assert rand_e == n * 2
    assert ring_e < full_e and star_e < full_e
    print(f"edges full={full_e} ring={ring_e} star={star_e} random_k={rand_e}")

    # 收敛坍塌：全员初始错误；无强制分歧 + 全连接会锁死错答
    collapse = DebateRuntime(n=4, personas=[f"p{i}" for i in range(4)], topology="full", seed=0)
    bad = ["Lyon", "Lyon", "Lyon", "Lyon"]
    r0 = collapse.run(
        "法国的首都是哪里？",
        rounds=2,
        forced_dissent_rounds=0,
        initial_answers=bad,
    )
    assert r0.final_answer == "Lyon"
    print("collapse without dissent -> Lyon (expected)")

    # 强制分歧：至少一轮必须反方案，随后批判可收敛到 Paris（脚本策略）
    r1 = collapse.run(
        "法国的首都是哪里？",
        rounds=2,
        forced_dissent_rounds=1,
        initial_answers=bad,
    )
    assert any(log.mode == "forced_dissent" for log in r1.logs)
    # 分歧后邻居众数变化，脚本策略会把多数拉向 Paris
    assert r1.final_answer == "Paris"
    print("forced dissent breaks collapse -> Paris")

    # 星形：hub 答案主导边缘可读集合
    star = DebateRuntime(n=4, personas=[f"p{i}" for i in range(4)], topology="star", seed=0)
    # hub 错、边缘对；无分歧时边缘只看见 hub → 易被污染
    init = ["Lyon", "Paris", "Paris", "Paris"]
    rs = star.run(
        "法国的首都是哪里？",
        rounds=1,
        forced_dissent_rounds=0,
        initial_answers=init,
    )
    # 边缘批判轮只读 hub=Lyon，会改成 Lyon；hub 读三人 Paris → Paris
    assert rs.agents[1].answer == "Lyon"
    print("star hub pollution on edges ok")

    # 异构 persona 长度
    rt = DebateRuntime(n=3, personas=["skeptic", "optimist", "historian"], topology="ring")
    out = rt.run("法国的首都是哪里？", rounds=1, forced_dissent_rounds=0)
    assert len({a.persona for a in out.agents}) == 3
    assert out.total_reads == count_edges(3, "ring")  # 仅 1 轮批判
    print("TOY DEMO OK")


demo_debate_toy()


edges full=20 ring=10 star=8 random_k=10
collapse without dissent -> Lyon (expected)
forced dissent breaks collapse -> Paris
star hub pollution on edges ok
TOY DEMO OK


## 3. 生产级：LangGraph 辩论图 + DeepSeek

状态机：`propose` → `forced_dissent`（可 0 次）→ `critique`×R → `vote`。  
稀疏拓扑决定每人上下文里塞哪些同伴答案；persona 列表保证提示词异构。需 `DEEPSEEK_API_KEY`。


In [9]:
import json
import os
import sys
from pathlib import Path
from typing import Any, Literal

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"


class DebateState(TypedDict, total=False):
    """LangGraph 辩论状态。"""

    question: str
    answers: list[str]
    personas: list[str]
    topology: Topology
    k: int
    round_idx: int
    max_critique_rounds: int
    forced_left: int
    total_reads: int
    phase: Literal["propose", "forced_dissent", "critique", "done"]
    final_answer: str
    log: list[dict[str, Any]]


def get_llm(*, temperature: float = 0.2) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def _invoke_text(prompt: str, *, temperature: float = 0.2) -> str:
    msg = get_llm(temperature=temperature).invoke(prompt)
    return str(msg.content).strip()


def propose_node(state: DebateState) -> dict[str, Any]:
    """
    各 persona 独立提案（不读同伴）。

    Returns:
        update: answers / phase / log。
    """
    q = state["question"]
    personas = list(state["personas"])
    answers: list[str] = []
    for p in personas:
        prompt = (
            f"You are debater persona: {p}.\n"
            f"Answer the question in ONE short phrase (no explanation).\n"
            f"Question: {q}\n"
            f"Answer:"
        )
        answers.append(_invoke_text(prompt, temperature=0.4))
    log = list(state.get("log") or [])
    log.append({"phase": "propose", "answers": answers, "reads": 0})
    return {
        "answers": answers,
        "phase": "forced_dissent" if int(state.get("forced_left") or 0) > 0 else "critique",
        "round_idx": 0,
        "total_reads": 0,
        "log": log,
    }


def _update_all(state: DebateState, *, mode: Literal["forced_dissent", "critique"]) -> dict[str, Any]:
    q = state["question"]
    answers = list(state["answers"])
    personas = list(state["personas"])
    n = len(answers)
    topo: Topology = state.get("topology") or "ring"
    k = int(state.get("k") or 2)
    rng = random.Random(hash(q) % (2**32))
    new_answers: list[str] = []
    reads = 0
    for i in range(n):
        nbrs = neighbors(n, i, topo, k=k, rng=rng)
        reads += len(nbrs)
        peer_block = "\n".join(f"- agent{j} ({personas[j]}): {answers[j]}" for j in nbrs)
        if mode == "forced_dissent":
            rule = (
                "FORCED DISSENT ROUND: you MUST disagree with the peers' majority. "
                "Give an alternative short answer. Do NOT say you agree."
            )
        else:
            rule = (
                "CRITIQUE ROUND: read peers, keep or revise your answer. "
                "One short phrase only."
            )
        prompt = (
            f"You are debater persona: {personas[i]}.\n"
            f"{rule}\n"
            f"Question: {q}\n"
            f"Your current answer: {answers[i]}\n"
            f"Peers you can see:\n{peer_block or '(none)'}\n"
            f"New answer (short phrase only):"
        )
        new_answers.append(_invoke_text(prompt, temperature=0.3 if mode == "critique" else 0.5))
    log = list(state.get("log") or [])
    log.append({"phase": mode, "answers": new_answers, "reads": reads})
    total = int(state.get("total_reads") or 0) + reads
    return {"answers": new_answers, "total_reads": total, "log": log}


def forced_dissent_node(state: DebateState) -> dict[str, Any]:
    """强制分歧一轮。"""
    upd = _update_all(state, mode="forced_dissent")
    left = max(0, int(state.get("forced_left") or 0) - 1)
    upd["forced_left"] = left
    upd["round_idx"] = int(state.get("round_idx") or 0) + 1
    upd["phase"] = "forced_dissent" if left > 0 else "critique"
    return upd


def critique_node(state: DebateState) -> dict[str, Any]:
    """普通批判一轮。"""
    upd = _update_all(state, mode="critique")
    upd["round_idx"] = int(state.get("round_idx") or 0) + 1
    done_crit = int(upd["round_idx"])  # simplified; track critique count via log
    # 用 log 里 critique 次数判断
    critiques = sum(1 for x in upd["log"] if x["phase"] == "critique")
    max_r = int(state.get("max_critique_rounds") or 1)
    upd["phase"] = "done" if critiques >= max_r else "critique"
    return upd


def vote_node(state: DebateState) -> dict[str, Any]:
    """多数票汇总。"""
    final = majority_vote(list(state["answers"]))
    log = list(state.get("log") or [])
    log.append({"phase": "vote", "final": final, "answers": list(state["answers"])})
    return {"final_answer": final, "phase": "done", "log": log}


def _route_after_propose(state: DebateState) -> str:
    """propose 之后只可能进入强制分歧或批判。"""
    return "forced_dissent" if (state.get("phase") == "forced_dissent") else "critique"


def _route_after_forced(state: DebateState) -> str:
    """强制分歧可自环，否则进入批判。"""
    return "forced_dissent" if (state.get("phase") == "forced_dissent") else "critique"


def _route_after_critique(state: DebateState) -> str:
    """批判可自环，否则投票。"""
    return "critique" if (state.get("phase") == "critique") else "vote"


def build_debate_graph() -> Any:
    """
    Returns:
        graph: 编译后的辩论状态图。

    每个节点单独注册可达 path map，避免把走不到的条件边画进图里。
    """
    g = StateGraph(DebateState)
    g.add_node("propose", propose_node)
    g.add_node("forced_dissent", forced_dissent_node)
    g.add_node("critique", critique_node)
    g.add_node("vote", vote_node)
    g.add_edge(START, "propose")
    g.add_conditional_edges(
        "propose",
        _route_after_propose,
        {"forced_dissent": "forced_dissent", "critique": "critique"},
    )
    g.add_conditional_edges(
        "forced_dissent",
        _route_after_forced,
        {"forced_dissent": "forced_dissent", "critique": "critique"},
    )
    g.add_conditional_edges(
        "critique",
        _route_after_critique,
        {"critique": "critique", "vote": "vote"},
    )
    g.add_edge("vote", END)
    return g.compile()


DEBATE_GRAPH = build_debate_graph()


def run_debate_impl(
    question: str,
    *,
    topology: Topology = "ring",
    k: int = 2,
    critique_rounds: int = 1,
    forced_dissent_rounds: int = 1,
    personas: list[str] | None = None,
) -> str:
    """
    跑一场稀疏拓扑辩论。

    Returns:
        json: final + answers + reads + log 摘要。
    """
    personas = personas or ["skeptic", "optimist", "historian"]
    init: DebateState = {
        "question": question,
        "answers": [],
        "personas": personas,
        "topology": topology,
        "k": k,
        "round_idx": 0,
        "max_critique_rounds": critique_rounds,
        "forced_left": forced_dissent_rounds,
        "total_reads": 0,
        "phase": "propose",
        "final_answer": "",
        "log": [],
    }
    out = DEBATE_GRAPH.invoke(init)
    return json.dumps(
        {
            "final_answer": out.get("final_answer"),
            "answers": out.get("answers"),
            "total_reads": out.get("total_reads"),
            "topology": topology,
            "phases": [x.get("phase") for x in (out.get("log") or [])],
            "log": out.get("log"),
        },
        ensure_ascii=False,
    )


class DebateArgs(BaseModel):
    question: str
    topology: Topology = "ring"
    critique_rounds: int = 1
    forced_dissent_rounds: int = 1


def build_control_tools() -> list[StructuredTool]:
    def _run(**kwargs: Any) -> str:
        a = DebateArgs(**kwargs)
        return run_debate_impl(
            a.question,
            topology=a.topology,
            critique_rounds=a.critique_rounds,
            forced_dissent_rounds=a.forced_dissent_rounds,
        )

    return [
        StructuredTool.from_function(
            name="run_debate",
            description="Run multi-agent debate with sparse topology and optional forced dissent.",
            func=_run,
            args_schema=DebateArgs,
        )
    ]


CONTROL_TOOLS = build_control_tools()


def build_control_agent() -> Any:
    system = (
        "You operate a multi-agent debate runtime.\n"
        "Use run_debate for non-trivial reasoning questions.\n"
        "Explain topology and whether forced dissent fired. Chinese."
    )
    return create_agent(get_llm(), CONTROL_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args') or {}})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 800 else str(m.content)[:800] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


print(f"LangGraph debate ready | {MODEL}")


LangGraph debate ready | deepseek:deepseek-v4-flash


## 3.1 可视化：辩论状态图

下一格用 **matplotlib** 出图（Cursor 对 `display(Markdown(mermaid))` 经常不渲染，所以你会只看到文字边列表）。

| 样式 | 含义 |
|---|---|
| 绿色实线 `fixed` | `add_edge` |
| 橙色虚线 `cond` | `add_conditional_edges`（已去掉不可达分支） |


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.lines import Line2D


def show_debate_graph(graph=DEBATE_GRAPH) -> None:
    """
    Draw fixed vs conditional edges with matplotlib (Mermaid often won't render in Cursor).

    Args:
        graph: compiled result of ``build_debate_graph()``.
    """
    g = graph.get_graph()
    G = nx.DiGraph()
    labels: dict[str, str] = {}
    for nid, node in g.nodes.items():
        name = getattr(node, "name", str(nid))
        G.add_node(str(nid))
        labels[str(nid)] = name

    fixed_edges: list[tuple[str, str]] = []
    cond_edges: list[tuple[str, str]] = []
    for e in g.edges:
        s, t = str(e.source), str(e.target)
        G.add_edge(s, t)
        (cond_edges if e.conditional else fixed_edges).append((s, t))

    pos = {
        "__start__": (0.0, 2.0),
        "propose": (1.6, 2.0),
        "forced_dissent": (3.4, 3.1),
        "critique": (3.4, 0.9),
        "vote": (5.2, 2.0),
        "__end__": (6.8, 2.0),
    }
    for n in G.nodes:
        pos.setdefault(n, (0.0, 0.0))

    fig, ax = plt.subplots(figsize=(11, 4.8))
    ax.set_title("Debate graph: green solid = fixed, orange dashed = conditional", fontsize=12)
    nx.draw_networkx_nodes(
        G,
        pos,
        ax=ax,
        node_size=2400,
        node_color="#EEF2FF",
        edgecolors="#334155",
        linewidths=1.2,
    )
    nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=9)

    nx.draw_networkx_edges(
        G,
        pos,
        edgelist=fixed_edges,
        ax=ax,
        edge_color="#15803d",
        width=2.6,
        arrows=True,
        arrowsize=22,
        connectionstyle="arc3,rad=0.0",
    )
    nx.draw_networkx_edges(
        G,
        pos,
        edgelist=cond_edges,
        ax=ax,
        edge_color="#ea580c",
        width=2.1,
        style="dashed",
        arrows=True,
        arrowsize=22,
        connectionstyle="arc3,rad=0.2",
    )

    edge_labels: dict[tuple[str, str], str] = {}
    for s, t in fixed_edges:
        edge_labels[(s, t)] = "fixed"
    for s, t in cond_edges:
        edge_labels[(s, t)] = "cond/self" if s == t else "cond"
    nx.draw_networkx_edge_labels(
        G, pos, edge_labels=edge_labels, ax=ax, font_size=8, font_color="#0f172a"
    )

    legend = [
        Line2D([0], [0], color="#15803d", lw=2.6, label="fixed = add_edge"),
        Line2D([0], [0], color="#ea580c", lw=2.1, ls="--", label="cond = add_conditional_edges"),
    ]
    ax.legend(handles=legend, loc="upper left", frameon=True)
    ax.set_axis_off()
    fig.tight_layout()
    display(fig)
    plt.close(fig)

    print("fixed edges:")
    for s, t in fixed_edges:
        print(f"  [fixed] {labels.get(s, s)} --> {labels.get(t, t)}")
    print("conditional edges (reachable only):")
    for s, t in cond_edges:
        mark = "loop" if s == t else "-->"
        print(f"  [cond]  {labels.get(s, s)} {mark} {labels.get(t, t)}")


show_debate_graph()


## 4. 生产示例：环拓扑辩论 + 强制分歧

无 `DEEPSEEK_API_KEY` 则 SKIP。


In [11]:
def demo_production_debate() -> None:
    """生产：环拓扑 + 强制分歧；断言 log 相位与可读边。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production: DEEPSEEK_API_KEY missing")
        return

    raw = run_debate_impl(
        "用一个词回答：水在标准大气压下的沸点（摄氏度数字即可）",
        topology="ring",
        critique_rounds=1,
        forced_dissent_rounds=1,
        personas=["chemist", "skeptic", "student"],
    )
    payload = json.loads(raw)
    print("=== debate ===")
    print(json.dumps({k: payload[k] for k in ("final_answer", "answers", "total_reads", "phases")}, ensure_ascii=False, indent=2))
    assert "forced_dissent" in payload["phases"]
    assert "critique" in payload["phases"]
    assert "vote" in payload["phases"]
    # 环：3 agent × 2 邻居 × (1 dissent + 1 critique) = 12
    assert payload["total_reads"] == 3 * 2 * 2
    print("PROD DEMO OK")


demo_production_debate()


=== debate ===
{
  "final_answer": "100",
  "answers": [
    "99.97",
    "100",
    "99.9"
  ],
  "total_reads": 12,
  "phases": [
    "propose",
    "forced_dissent",
    "critique",
    "vote"
  ]
}
PROD DEMO OK
